#### **Installing dependencies**

In [1]:
!pip install transformers sentencepiece datasets tqdm rouge -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.12.0 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.8.4.1 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cudnn-cu12==9.1.0.70; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cudnn-cu12 9.3.0.75 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cufft-cu12==11.2.1.3; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cufft-cu12 11.3.3.83 which is incompatible.
torch 2.5.1+cu124 requires nvidia-curand-cu12==10.3.5.147; platform_system == "Linux" and platform_machin

In [2]:
import torch
import pandas as pd
from rouge import Rouge
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import plotly.express as px
import matplotlib.pyplot as plt

from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)

In [3]:
from datasets import load_dataset
ds = load_dataset("csebuetnlp/xlsum", "hindi", split='all')
data = ds.to_pandas()
data = data.sample(frac = 0.125)

README.md:   0%|          | 0.00/14.6k [00:00<?, ?B/s]

xlsum.py:   0%|          | 0.00/4.55k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/187M [00:00<?, ?B/s]

0001.parquet:   0%|          | 0.00/19.6M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/21.5M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/70778 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8847 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8847 [00:00<?, ? examples/s]

In [4]:
data = data.drop(['id', 'url'], axis=1)
data


,title,summary,text
62032,मोदी का उपग्रह के ज़रिए संबोधन,गुजरात के मुख्यमंत्री नरेंद्र मोदी ने अमरीकी न...,ग़ौरतलब है कि अमरीका ने पिछले सप्ताह नरेंद्र म...
41064,कोरोना वायरस: लॉकडाउन में क्या बंद और क्या खुल...,जनता कर्फ़्यू को गंभीरता से लेने पर प्रधानमंत्...,उन्होंने ट्वीट किया है कि 'देशवासियों ने बता द...
52698,कलाम ने बच्चों से सपने देखने को कहा,भारत के राष्ट्रपति डॉ. एपीजे अब्दुल कलाम ने मध...,"उन्होंने कहा कि सड़क, बिजली और कंप्यूटर जैसी स..."
58868,"ऑस्ट्रेलिया दवाब में, पाँच विकेट गिरे",मोहाली टेस्ट के चौथे दिन ऑस्ट्रेलिया ने 516 रन...,सोमवार को भारत ने अपनी दूसरी पारी कुल 314 रन ब...
53989,'स्पिन आक्रमण के लिए तैयार हैं',टेस्ट मैच शृंखला के लिए भारत पहुंची ऑस्ट्रेलिय...,भारत के दौरे में रिकी पोंटिंग अक्सर स्पिन गेंद...
...,...,...,...
61977,कोविड से बेटे की मौत पर बोले सीताराम येचुरी- म...,कम्युनिस्ट पार्टी ऑफ इंडिया (मार्क्सवादी) यानी...,"अपने ट्वीट में येचुरी ने डॉक्टरों, नर्सों, फ्र..."
42256,पीएम मोदी के कोलकाता दौरे से राजनीति गर्म,नेशनल रजिस्टर ऑफ़ सिटिज़नशिप (एनआरसी) और नागरि...,मोदी शनिवार को कोलकाता आएंगे. वाम दलों और कांग...
25978,अमिताभ को टैक्स चोरी का नोटिस,सुप्रीम कोर्ट ने टैक्स चुराने के मामले में बॉल...,आयकर विभाग का कहना है कि अमिताभ ने टैक्स बचाया...
26697,"आज की पाँच बड़ी ख़बरें: रफ़ाल पर नई जानकारी, स...",भारत और फ़्रांस के बीच हुए रफ़ाल सौदे को लेकर ...,मीडियापार्ट ने डासो एविएशन के आंतरिक दस्तावेज़...


In [5]:
text = data['text']
goldsummary = data['summary']

### **Model1 : BART**

In [6]:
bart_checkpoint = "ai4bharat/IndicBART"

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [8]:
tokenizer_bart = AutoTokenizer.from_pretrained(bart_checkpoint, 
                                               do_lower_case=False, 
                                               use_fast=False, 
                                               keep_accents=True)

model_bart = AutoModelForSeq2SeqLM.from_pretrained(bart_checkpoint).to(device)

tokenizer_config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.90M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/221 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

2025-04-24 21:55:11.721881: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745531711.922959      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745531711.978545      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


pytorch_model.bin:   0%|          | 0.00/976M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/976M [00:00<?, ?B/s]

In [9]:
bos_id = tokenizer_bart._convert_token_to_id_with_added_voc("<s>")
eos_id = tokenizer_bart._convert_token_to_id_with_added_voc("</s>")
pad_id = tokenizer_bart._convert_token_to_id_with_added_voc("<pad>")

In [10]:
%%time

from tqdm import tqdm

SystemSummary = []

for i, input_text in tqdm(enumerate(text), total=len(text), desc="Generating Summaries"):
    inp = tokenizer_bart(input_text, add_special_tokens=False, 
                         truncation=True, return_tensors="pt", 
                         padding='max_length', max_length=1024)['input_ids'].to(device)  
    
    model_output = model_bart.generate(inp, use_cache=True, 
                                        num_beams=4, 
                                        max_length=70, 
                                        min_length=30, 
                                        early_stopping=True, 
                                        pad_token_id=pad_id,
                                        bos_token_id=bos_id, 
                                        eos_token_id=eos_id, 
                                        decoder_start_token_id=tokenizer_bart._convert_token_to_id_with_added_voc("<2en>"))
    
    decoded_output = tokenizer_bart.decode(model_output[0], 
                                    skip_special_tokens=True, 
                                    clean_up_tokenization_spaces=False)
    
    SystemSummary.append(decoded_output)


Generating Summaries: 100%|██████████| 11059/11059 [1:35:27<00:00,  1.93it/s]

CPU times: user 1h 35min 28s, sys: 11.8 s, total: 1h 35min 39s
Wall time: 1h 35min 27s


In [11]:
Summaries = pd.DataFrame(list(zip(goldsummary, SystemSummary)), columns =['GoldSummary', 'BartSummary'])
Summaries

,GoldSummary,BartSummary
0,गुजरात के मुख्यमंत्री नरेंद्र मोदी ने अमरीकी न...,ग़ौरतलब है कि अमरीका ने पिछले सप्ताह नरेंद्र म...
1,जनता कर्फ़्यू को गंभीरता से लेने पर प्रधानमंत्...,उन्होंने ट्वीट किया है कि 'देशवासियों ने बता द...
2,भारत के राष्ट्रपति डॉ. एपीजे अब्दुल कलाम ने मध...,"उन्होंने कहा कि सड़क, बिजली और कंप्यूटर जैसी स..."
3,मोहाली टेस्ट के चौथे दिन ऑस्ट्रेलिया ने 516 रन...,सोमवार को भारत ने अपनी दूसरी पारी कुल 314 रन ब...
4,टेस्ट मैच शृंखला के लिए भारत पहुंची ऑस्ट्रेलिय...,भारत के दौरे में रिकी पोंटिंग अक्सर स्पिन गेंद...
...,...,...
11054,कम्युनिस्ट पार्टी ऑफ इंडिया (मार्क्सवादी) यानी...,"अपने ट्वीट में येचुरी ने डॉक्टरों, नर्सों, फ्र..."
11055,नेशनल रजिस्टर ऑफ़ सिटिज़नशिप (एनआरसी) और नागरि...,मोदी शनिवार को कोलकाता आएंगे. वाम दलों और कांग...
11056,सुप्रीम कोर्ट ने टैक्स चुराने के मामले में बॉल...,आयकर विभाग का कहना है कि अमिताभ ने टैक्स बचाया...
11057,भारत और फ़्रांस के बीच हुए रफ़ाल सौदे को लेकर ...,मीडियापार्ट ने डासो एविएशन के आंतरिक दस्तावेज़...


In [12]:
rouge = Rouge()
score = rouge.get_scores(Summaries['BartSummary'], Summaries['GoldSummary'], avg=True)
BartRouge = pd.DataFrame(score).set_index([['recall','precision','f-measure']])
BartRouge

,rouge-1,rouge-2,rouge-l
recall,0.373179,0.093419,0.315025
precision,0.204923,0.045523,0.172315
f-measure,0.259175,0.059797,0.218148


### **Model2 : T5**

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
t5_checkpoint = "csebuetnlp/mT5_multilingual_XLSum"

tokenizer_t5 = AutoTokenizer.from_pretrained(t5_checkpoint)
model_t5 = AutoModelForSeq2SeqLM.from_pretrained(t5_checkpoint).to(device)

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:559: UserWarning:

The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.



pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

In [14]:
%%time

from tqdm import tqdm

SystemSummary1 = []

for i, input_text in tqdm(enumerate(text), total=len(text), desc="Generating Summaries"):
    input_ids = tokenizer_t5(input_text,
                             return_tensors="pt",
                             padding="max_length",
                             truncation=True,
                             max_length=768)["input_ids"].to(device) 

    model_output = model_t5.generate(input_ids=input_ids,
                                     num_beams=4,
                                     max_length=70,
                                     min_length=30,
                                     no_repeat_ngram_size=2,
                                     early_stopping=True)

    decoded_output = tokenizer_t5.decode(model_output[0],
                                  skip_special_tokens=True,
                                  clean_up_tokenization_spaces=False)
    
    SystemSummary1.append(decoded_output)


Generating Summaries: 100%|██████████| 11059/11059 [3:33:52<00:00,  1.16s/it]

CPU times: user 3h 33min 58s, sys: 13.2 s, total: 3h 34min 11s
Wall time: 3h 33min 52s


In [15]:
Summaries['T5Summary'] = SystemSummary1

In [16]:
Summaries.head()

,GoldSummary,BartSummary,T5Summary
0,गुजरात के मुख्यमंत्री नरेंद्र मोदी ने अमरीकी न...,ग़ौरतलब है कि अमरीका ने पिछले सप्ताह नरेंद्र म...,भारत के प्रधानमंत्री नरेंद्र मोदी ने अमरीका मे...
1,जनता कर्फ़्यू को गंभीरता से लेने पर प्रधानमंत्...,उन्होंने ट्वीट किया है कि 'देशवासियों ने बता द...,भारत के प्रधानमंत्री नरेंद्र मोदी ने कहा है कि...
2,भारत के राष्ट्रपति डॉ. एपीजे अब्दुल कलाम ने मध...,"उन्होंने कहा कि सड़क, बिजली और कंप्यूटर जैसी स...",भारत के राष्ट्रपति एपीजे अब्दुल कलाम ने कहा है...
3,मोहाली टेस्ट के चौथे दिन ऑस्ट्रेलिया ने 516 रन...,सोमवार को भारत ने अपनी दूसरी पारी कुल 314 रन ब...,भारत और ऑस्ट्रेलिया के बीच सिडनी में खेले जा र...
4,टेस्ट मैच शृंखला के लिए भारत पहुंची ऑस्ट्रेलिय...,भारत के दौरे में रिकी पोंटिंग अक्सर स्पिन गेंद...,ऑस्ट्रेलियाई क्रिकेट टीम के कप्तान रिकी पोंटिं...


In [17]:
rouge = Rouge()
score = rouge.get_scores(Summaries['T5Summary'], Summaries['GoldSummary'], avg=True)
T5Rouge = pd.DataFrame(score).set_index([['recall','precision','f-measure']])
T5Rouge

,rouge-1,rouge-2,rouge-l
recall,0.347126,0.146683,0.300180
precision,0.437316,0.191032,0.377572
f-measure,0.376852,0.160993,0.325595


### **Plotting Rouge Score**

In [18]:
T5Rouge = T5Rouge[['rouge-1','rouge-l', 'rouge-2']]
BartRouge = BartRouge[['rouge-1','rouge-l', 'rouge-2']]

In [19]:
fig = px.bar(T5Rouge*100, x=T5Rouge.index, y=T5Rouge.columns, 
             barmode='group', 
             text_auto='.2s',
             labels={
                     "Algo": "Algorithms",
                     "value": "Rouge Score",
                     "variable": "legend",
                     'index': "Metrics"
                 })

fig.update_layout(width=650,
                  height=400,
                  title={
                  'text': "Score",
                  'y':.96,
                  'x':0.49,
                  'xanchor': 'center',
                  'yanchor': 'top'})

fig.show()

In [20]:
fig = px.bar(BartRouge*100, x=BartRouge.index, y=BartRouge.columns, 
             barmode='group', 
             text_auto='.2s',
             labels={
                     "Algo": "Algorithms",
                     "value": "Rouge Score",
                     "variable": "legend",
                     'index': "Metrics"
                 })
fig.update_layout( width=650,
                  height=400,
                title={
                  'text': "Score",
                  'y':.96,
                  'x':0.49,
                  'xanchor': 'center',
                  'yanchor': 'top'})

fig.show()

### **Sample Summaries**

In [21]:
[i for i in Summaries['GoldSummary'][:5]]

['गुजरात के मुख्यमंत्री नरेंद्र मोदी ने अमरीकी न्यूयॉर्क में एक जनसभा को उपग्रह के ज़रिए संबोधित किया है.',
 'जनता कर्फ़्यू को गंभीरता से लेने पर प्रधानमंत्री नरेंद्र मोदी ने भारत के लोगों की तारीफ़ की है.',
 'भारत के राष्ट्रपति डॉ. एपीजे अब्दुल कलाम ने मध्यप्रदेश के ईटखेड़ी गाँव के बच्चों से कहा कि वो शहरी और ग्रामीण जीवन के अंतर को दूर करने का सपना देखें.',
 'मोहाली टेस्ट के चौथे दिन ऑस्ट्रेलिया ने 516 रनों के लक्ष्य का पीछा करते हुए पाँच विकेट पर 141 रन बनाए लिए हैं.',
 'टेस्ट मैच शृंखला के लिए भारत पहुंची ऑस्ट्रेलियाई टीम के कप्तान रिकी पोंटिंग का कहना है कि वो भारत के स्पिन आक्रमण के लिए पूरी तरह तैयार हैं.']

In [22]:
[i for i in Summaries['T5Summary'][:5]]

['भारत के प्रधानमंत्री नरेंद्र मोदी ने अमरीका में एक जनसभा को संबोधित करते हुए कहा है कि माँ भारती के माथे पर काला टीका लगाया गया है.',
 'भारत के प्रधानमंत्री नरेंद्र मोदी ने कहा है कि देश में कोरोना वायरस संक्रमण के मामले बढ़ रहे हैं.',
 'भारत के राष्ट्रपति एपीजे अब्दुल कलाम ने कहा है कि अंतरिक्ष विज्ञान के क्षेत्र में बच्चों को सुविधाएँ मिलनी चाहिए.',
 'भारत और ऑस्ट्रेलिया के बीच सिडनी में खेले जा रहे दूसरे टेस्ट मैच में भारत ने पहली पारी में 131 रन बना लिए हैं.',
 'ऑस्ट्रेलियाई क्रिकेट टीम के कप्तान रिकी पोंटिंग का कहना है कि वो भारत के दौरे के लिए स्पिन गेंदबाज़ी करेंगे.']

In [23]:
[i for i in Summaries['BartSummary'][:5]]

['ग़ौरतलब है कि अमरीका ने पिछले सप्ताह नरेंद्र मोदी को यह कहते हुए वीज़ा देने से इनकार कर दिया था कि मोदी ने धार्मिक स्वतंत्रता का उल्लंघन किया है. मोदी को वीज़ा नहीं दिए जाने के मुद्दे ने काफ़ी तूल पकड़ा था और भारत सरकार ने इस मुद्दे पर अमरीका सरकार से नाराज़गी जताई थी. नरेंद्र मोदी अमरीका तो नहीं जा',
 "उन्होंने ट्वीट किया है कि 'देशवासियों ने बता दिया कि हम सक्षम हैं. लेकिन यह एक लंबी लड़ाई की शुरुआत है.' पीएम मोदी ने दो दिन पहले, 22 मार्च को एक दिवसीय कर्फ़्यू लगाने की घोषणा की थी. लेकिन इस 'लड़ाई' की गंभीरता को देखते हुए भारत सरकार ने रविवार शाम देश के 22 राज्यों",
 'उन्होंने कहा कि सड़क, बिजली और कंप्यूटर जैसी सुविधाएँ गाँवों को भी मिलनी चाहिए. वह मंगलवार को भोपाल से नौ किलोमीटर दूर ईटखेड़ी के एक स्कूल के दौरे पर थे. राष्ट्रपति कलाम ने बच्चों से दोहराने को कहा, "सपने, सपने, सपने...सपनों को विचारों में बदलिए...और',
 'सोमवार को भारत ने अपनी दूसरी पारी कुल 314 रन बनाकर पारी घोषित कर दी थी. इस तरह अब ऑस्ट्रेलिया को जीत के लिए कुल 516 बनाने होंगे. ज़ाहिर है अब दबाव ऑस्ट्रेलिया की टीम 